# Simultaneous dengue surges — full multi-region experiment v2
**Kaggle: enable Internet, use CPU, then Run All.** No API keys or uploaded datasets are needed. Downloads can be slow; progress is printed for each municipality. The last cell exports `/kaggle/working/dengue_full_results.zip`, including raw data and all results.

**Scientific question:** Does accounting for cross-city dependence improve forecasts of the number of simultaneously surging municipalities and improve explicitly hypothetical reserve decisions?

**Scope:** six fixed 10-municipality geographic study panels; four development panels (including the previously examined Rio pilot) and two external panels (Fortaleza and Recife). The panels are named geographic selections, **not verified top-10 population rankings or complete official metropolitan areas**. Verify their operational relevance before manuscript submission. No municipality is replaced on the basis of outcomes.

**What changed from the pilot:** separated marginal-calibration and dependence-estimation blocks; calibration support gates; calibrated and raw forecasts have distinct names; full 0–10 surge-count distributions; matched marginal probabilities for independent/dependent aggregation; ordinal count model; chronological availability purging; external geographic evaluation; online resource budgets; paired time-block comparisons; explicit quality, sensitivity and failure reports.

This is a **retrospective revised-data study**, not a real-time historical backtest. InfoDengue updates past cases. No archive of as-issued data is supplied here. No novelty, acceptance, hospital-overload or achieved health-benefit claim follows automatically.

Source documentation: https://info.dengue.mat.br/tutorial_api_python/ and https://info.dengue.mat.br/services/api/doc . Counts are surveillance reports, not necessarily laboratory-confirmed infections.

The code has fixed model settings rather than an automatic algorithm search. The core matched comparison is established statistical methodology, not a claimed new ML algorithm. Full execution on the six live panels has **not** been performed while preparing this file, to avoid examining the external outcomes. Verification used explicitly synthetic test fixtures and the previously available Rio snapshot; synthetic fixtures are not included or substituted in this research notebook.

## 1. Freeze the design and environment
Primary endpoint: equal-region normalized ranked probability score (RPS) for the 0–10 count distribution at two weeks, comparing shared-factor versus independent aggregation of **identical calibrated logistic city probabilities**, in the two external regions. Brier score for ≥3 surges and hypothetical cost are secondary. One- and four-week horizons and the boosting family are secondary.

`run_external=True` executes the fixed final external evaluation after development checks. Set it to False **before the first run** if you only want development results. Once external results have been examined, changing the method requires a new evaluation strategy; the same data cannot be called untouched again. The notebook saves a protocol hash before downloading data and refuses incompatible reuse of the output directory.

The three chronological stages are model fitting → marginal/direct calibration (A) → dependence estimation (B) → evaluation. Every training/calibration target must be observable before the next stage starts under the assumed reporting lag. No future weather or final annual totals are predictors. Fixed model settings give every method zero hyperparameter-search trials.

In [ ]:
import os, sys, io, json, time, hashlib, shutil, traceback, platform, warnings
from pathlib import Path
from datetime import datetime, timezone
from urllib.request import Request, urlopen
from urllib.parse import urlencode
from concurrent.futures import ThreadPoolExecutor, as_completed
import importlib.metadata as md
import unicodedata
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, FileLink
from scipy.special import ndtr, ndtri, expit, logsumexp
from scipy.optimize import minimize, minimize_scalar
from numpy.polynomial.hermite import hermgauss
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss

IMPLEMENTATION_SHA256='c6076e7c20c58f3c7e0c5e672b4d5f7f5fee5568d509507b4d0a0988d7b8fa4b'
# Freeze before accessing external-region outcomes. Do not tune from final results.
CFG = dict(seed=42, start_year=2015, end_year=2025, primary_horizon=2,
           horizons=[1,2,4], reporting_lag=2, quantile=.90, incidence_floor=10., k=3,
           min_coverage=.95, bootstrap_reps=1000, block_weeks=13, download_workers=3,
           timeout=60, retries=3, min_cal_positives=20, min_cal_negatives=20,
           min_dependence_weeks=52, quadrature_nodes=31, rho_max=.8,
           rho_penalty=.02, run_external=True, run_sensitivity=True,
           strict_quality=True, max_model_iterations=120)

# Fixed geographic study panels, NOT a verified population ranking or complete metro membership.
# Codes are checked against returned municipality names when the API provides them.
REGIONS = {
 'Rio': dict(role='development_pilot_seen', cities={
  '3304557':'Rio de Janeiro','3303302':'Niteroi','3304904':'Sao Goncalo',
  '3301702':'Duque de Caxias','3303500':'Nova Iguacu','3300456':'Belford Roxo',
  '3302858':'Mesquita','3303203':'Nilopolis','3304144':'Queimados','3305109':'Sao Joao de Meriti'}),
 'Sao_Paulo': dict(role='development', cities={
  '3550308':'Sao Paulo','3518800':'Guarulhos','3548708':'Sao Bernardo do Campo',
  '3547809':'Santo Andre','3534401':'Osasco','3513801':'Diadema','3529401':'Maua',
  '3510609':'Carapicuiba','3523107':'Itaquaquecetuba','3552502':'Suzano'}),
 'Belo_Horizonte': dict(role='development', cities={
  '3106200':'Belo Horizonte','3118601':'Contagem','3106705':'Betim',
  '3154606':'Ribeirao das Neves','3157807':'Santa Luzia','3129806':'Ibirite',
  '3156700':'Sabara','3171204':'Vespasiano','3137601':'Lagoa Santa','3149309':'Pedro Leopoldo'}),
 'Curitiba': dict(role='development', cities={
  '4106902':'Curitiba','4125506':'Sao Jose dos Pinhais','4105805':'Colombo',
  '4101804':'Araucaria','4119152':'Pinhais','4104204':'Campo Largo',
  '4100400':'Almirante Tamandare','4119509':'Piraquara','4107652':'Fazenda Rio Grande',
  '4104006':'Campina Grande do Sul'}),
 'Fortaleza': dict(role='external', cities={
  '2304400':'Fortaleza','2303709':'Caucaia','2307650':'Maracanau','2307700':'Maranguape',
  '2301000':'Aquiraz','2309706':'Pacatuba','2306256':'Itaitinga','2305233':'Horizonte',
  '2309607':'Pacajus','2304285':'Eusebio'}),
 'Recife': dict(role='external', cities={
  '2611606':'Recife','2607901':'Jaboatao dos Guararapes','2609600':'Olinda',
  '2610707':'Paulista','2606804':'Igarassu','2600054':'Abreu e Lima',
  '2603454':'Camaragibe','2613701':'Sao Lourenco da Mata','2602902':'Cabo de Santo Agostinho',
  '2607208':'Ipojuca'})}
DEV=[r for r,v in REGIONS.items() if v['role']!='external']
EXT=[r for r,v in REGIONS.items() if v['role']=='external']
assert all(len(v['cities'])==10 for v in REGIONS.values())
assert len({c for v in REGIONS.values() for c in v['cities']})==60
FOLDS=[
 dict(name='development_2021',train_end='2017-12-31',cal_a_start='2018-01-01',
      cal_a_end='2018-12-31',cal_b_start='2019-01-01',cal_b_end='2020-12-31',
      test_start='2021-01-01',test_end='2021-12-31'),
 dict(name='development_2022',train_end='2018-12-31',cal_a_start='2019-01-01',
      cal_a_end='2019-12-31',cal_b_start='2020-01-01',cal_b_end='2021-12-31',
      test_start='2022-01-01',test_end='2022-12-31'),
 dict(name='final',train_end='2018-12-31',cal_a_start='2019-01-01',
      cal_a_end='2020-12-31',cal_b_start='2021-01-01',cal_b_end='2022-12-31',
      test_start='2023-01-01',test_end='2025-12-31')]
BASE=Path('/kaggle/working') if Path('/kaggle/working').exists() else Path.cwd()
OUT=BASE/'dengue_full_outputs'
RAW,TABLES,FIGS=[OUT/p for p in ['raw','tables','figures']]
for p in [RAW,TABLES,FIGS]: p.mkdir(parents=True,exist_ok=True)
np.random.seed(CFG['seed']); sns.set_theme(style='whitegrid')
protocol=dict(config=CFG,regions=REGIONS,folds=FOLDS,
 implementation_sha256=IMPLEMENTATION_SHA256,
 primary_comparison='lr calibrated shared-factor vs identical calibrated independent marginals',
 primary_endpoint='equal-region mean ranked probability score, external regions, horizon=2',
 analysis_type='retrospective revised-data snapshot; no historical data vintages supplied',
 selection='geographically chosen fixed named panels; not population-ranked',
 context='Rio 2023-2025 pilot outcomes have already been reviewed')
protocol_bytes=json.dumps(protocol,sort_keys=True,indent=2).encode()
PROTOCOL_HASH=hashlib.sha256(protocol_bytes).hexdigest()
existing=OUT/'protocol.json'
if existing.exists() and hashlib.sha256(existing.read_bytes()).hexdigest()!=PROTOCOL_HASH:
    raise RuntimeError('Protocol changed in an existing output directory. Use a new clean output directory; preserve earlier results.')
existing.write_bytes(protocol_bytes)
(OUT/'protocol_sha256.txt').write_text(PROTOCOL_HASH)
(OUT/'environment.json').write_text(json.dumps(dict(python=sys.version,platform=platform.platform(),
 packages={p:md.version(p) for p in ['numpy','pandas','scipy','scikit-learn','matplotlib','seaborn']}),indent=2))
print('Protocol:',PROTOCOL_HASH,'\nOutput:',OUT)
print('External evaluation enabled:',CFG['run_external'])
display(pd.DataFrame([dict(region=r,role=v['role'],municipalities=len(v['cities'])) for r,v in REGIONS.items()]))


## 2. Download development panels only
Official API downloads are retried, cached with hashes and timestamp metadata, and validated against the expected CSV schema. Three requests at most run concurrently. A failed city stops the panel; it is not silently removed or replaced. On failure, run the final export cell for a diagnostic ZIP. Fresh runs should use a clean Kaggle working directory; cached files retain their original download metadata.

In [ ]:
API='https://info.dengue.mat.br/api/alertcity'
def normalized(s):
    return ''.join(c for c in unicodedata.normalize('NFKD',str(s)) if c.isalnum() and not unicodedata.combining(c)).casefold()

def parse_source(raw):
    text=raw.decode('utf-8-sig')
    if text.lstrip().lower().startswith(('<html','<!doctype')): raise ValueError('HTML returned instead of CSV')
    d=pd.read_csv(io.StringIO(text)); d.columns=d.columns.str.strip()
    if d.empty or not {'data_iniSE','casos','pop'}.issubset(d.columns):
        raise ValueError('Empty CSV or incompatible schema')
    return d

def fetch_one(region,code,name):
    params=dict(geocode=code,disease='dengue',format='csv',ew_start=1,ew_end=53,
                ey_start=CFG['start_year'],ey_end=CFG['end_year'])
    path=RAW/f'{code}_{CFG["start_year"]}_{CFG["end_year"]}.csv'
    meta=path.with_suffix('.json')
    if path.exists() and meta.exists():
        raw=path.read_bytes(); record=json.loads(meta.read_text())
        assert record['sha256']==hashlib.sha256(raw).hexdigest(),'Cached file hash mismatch'
    else:
        for attempt in range(CFG['retries']):
            try:
                req=Request(API+'?'+urlencode(params),headers={'User-Agent':'DengueMultiRegionResearch/2.0'})
                with urlopen(req,timeout=CFG['timeout']) as response: raw=response.read()
                parse_source(raw); break
            except Exception:
                if attempt==CFG['retries']-1: raise
                time.sleep(2**attempt)
        record=dict(region=region,geocode=code,expected_name=name,url=API+'?'+urlencode(params),
          downloaded_at_utc=datetime.now(timezone.utc).isoformat(),sha256=hashlib.sha256(raw).hexdigest(),bytes=len(raw))
        path.write_bytes(raw); meta.write_text(json.dumps(record,indent=2))
    d=parse_source(raw)
    if 'municipio_nome' in d and d['municipio_nome'].notna().any():
        observed=d['municipio_nome'].dropna().unique().tolist()
        if any(normalized(x)!=normalized(name) for x in observed):
            raise ValueError(f'Municipality name mismatch for {code}: expected {name}, got {observed}')
    d['region']=region; d['geocode']=code; d['municipality']=name
    return d,dict(record,rows=len(d))

def download_regions(region_names):
    jobs=[(r,c,n) for r in region_names for c,n in REGIONS[r]['cities'].items()]
    frames=[]; records=[]; errors=[]
    with ThreadPoolExecutor(max_workers=CFG['download_workers']) as pool:
        futures={pool.submit(fetch_one,*j):j for j in jobs}
        for future in as_completed(futures):
            r,c,n=futures[future]
            try:
                d,rec=future.result();frames.append(d);records.append(rec)
                print(r,n,len(d),'rows',flush=True)
            except Exception as exc:
                errors.append(dict(region=r,geocode=c,municipality=n,error=str(exc)))
                print('DOWNLOAD FAILED:',r,n,str(exc)[:150],flush=True)
    pd.DataFrame(records).to_csv(OUT/('manifest_'+region_names[0]+'.csv'),index=False)
    pd.DataFrame(errors,columns=['region','geocode','municipality','error']).to_csv(OUT/('failures_'+region_names[0]+'.csv'),index=False)
    if errors: raise RuntimeError('Incomplete fixed panel. No cities were silently replaced. Run final export cell for diagnostics.')
    return pd.concat(frames,ignore_index=True)

source_dev=download_regions(DEV)
source_dev.to_csv(OUT/'source_development.csv',index=False)


## 3. Quality audit and regular weekly panels
Missing values stay missing. No disease-count interpolation is used. Municipality totals are used for Rio when sublocality identifiers are present. Invalid counts/populations and duplicate weeks are exported. Coverage below 95% stops execution by default. Long zero runs are unresolved flags: they are neither treated as missing automatically nor certified as genuine no-transmission periods. Investigate these before a confirmatory manuscript.

In [ ]:
DATES=pd.date_range(f"{CFG['start_year']}-01-01",f"{CFG['end_year']}-12-31",freq='W-SUN')
PANELS={}; QUALITY=[]
def build_panels(source,region_names):
    for region in region_names:
        d=source[source.region.eq(region)].copy()
        d['date']=pd.to_datetime(d['data_iniSE'],errors='coerce').dt.normalize()
        if d.date.isna().any(): raise ValueError('Unparseable week dates')
        if 'Localidade_id' in d:
            loc=pd.to_numeric(d.Localidade_id,errors='coerce')
            rio=d.geocode.eq('3304557')
            if rio.any() and loc[rio].notna().any():
                if not loc[rio].eq(0).any(): raise ValueError('Rio municipality-level total cannot be identified')
                d=d.loc[~rio|loc.eq(0)].copy()
        d=d[d.date.isin(DATES)].copy()
        duplicates=d[d.duplicated(['geocode','date'],keep=False)]
        duplicates.to_csv(TABLES/f'duplicates_{region}.csv',index=False)
        if len(duplicates): raise ValueError(f'Duplicate municipality-weeks: {region}')
        for col in ['casos','pop']: d[col]=pd.to_numeric(d[col],errors='coerce')
        invalid=d.casos.lt(0)|d['pop'].le(0)|(~np.isclose(d.casos.fillna(0),d.casos.fillna(0).round()))
        d.loc[invalid].to_csv(TABLES/f'invalid_{region}.csv',index=False)
        d.loc[invalid,['casos','pop']]=np.nan
        d['incidence']=100000*d.casos/d['pop']
        index=pd.MultiIndex.from_product([list(REGIONS[region]['cities']),DATES],names=['geocode','date'])
        p=d.set_index(['geocode','date']).reindex(index).reset_index()
        p['region']=region;p['municipality']=p.geocode.map(REGIONS[region]['cities'])
        p['year']=p.date.dt.year
        p.to_csv(OUT/f'panel_{region}.csv',index=False)
        for code,g in p.groupby('geocode'):
            zeros=g.casos.eq(0)
            row=dict(region=region,geocode=code,municipality=REGIONS[region]['cities'][code],
              weeks=len(g),valid_weeks=g.incidence.notna().sum(),coverage=g.incidence.notna().mean(),
              zero_weeks=zeros.sum(),longest_zero_run=int(zeros.groupby((~zeros).cumsum()).sum().max()),
              population_jump_max=g['pop'].pct_change(fill_method=None).abs().max())
            QUALITY.append(row)
        inc=p.pivot(index='date',columns='geocode',values='incidence').reindex(index=DATES,columns=list(REGIONS[region]['cities']))
        PANELS[region]=inc
    q=pd.DataFrame(QUALITY);q.to_csv(TABLES/'quality_all_available.csv',index=False)
    yearly=pd.concat([pd.read_csv(OUT/f'panel_{r}.csv') for r in region_names]).groupby(
      ['region','municipality','year']).agg(valid_weeks=('incidence','count'),zero_weeks=('casos',lambda s:s.eq(0).sum()))
    yearly.to_csv(TABLES/f'annual_coverage_{region_names[0]}.csv')
    bad=q[q.region.isin(region_names)&q.coverage.lt(CFG['min_coverage'])]
    if len(bad) and CFG['strict_quality']:
        raise RuntimeError('Coverage gate failed. Export diagnostics; no automatic panel replacement.')
    return q

quality_dev=build_panels(source_dev,DEV)
display(quality_dev)
print('Long zero runs are unresolved surveillance-quality flags, not proof of absent transmission.')


## 4. Training-only reference thresholds and forecast construction
A municipality is surging if reported weekly incidence per 100,000 is **strictly greater than** `max(training 90th percentile, 10)`. This is an exploratory operational definition, not an official outbreak declaration. The target count is the number of the ten cities surging in the **same exact future week**, not different weeks within a window. The binary target is count ≥3.

Each fold defines thresholds using historical observations available before its first calibration block. External cities use the same rule on their historical reference period to define labels and normalize features; **those reference histories are permitted local information**, so this is geographic transfer with local historical reference thresholds, not no-history cold-start forecasting. External outcomes never fit an ML model, calibrator or dependence parameter.

City models use own histories plus symmetric regional summaries; direct regional models use those regional summaries. The matched independent/dependent pair uses precisely the same city probability matrix, avoiding a feature-information confound. Current alert levels, Rt, nowcasts, city IDs and absolute calendar year are excluded. Latest input week is origin−2 by default; target week is origin+h.

In [ ]:
THRESHOLD_LOG=[]; EVENT_SUPPORT=[]
def episodes_in(s):
    positive=s.eq(1)
    contiguous=s.index.to_series().diff().eq(pd.Timedelta(weeks=1)).to_numpy()
    previous=positive.shift(1,fill_value=False).to_numpy()&contiguous
    return int((positive.to_numpy()&~previous).sum())

def make_dataset(regions,fold,h,lag=None,q=None,floor=None):
    lag=CFG['reporting_lag'] if lag is None else lag
    q=CFG['quantile'] if q is None else q
    floor=CFG['incidence_floor'] if floor is None else floor
    scenario=f'q{q:g}_floor{floor:g}_lag{lag}'
    reference_end=min(pd.Timestamp(fold['train_end']),pd.Timestamp(fold['cal_a_start'])-pd.Timedelta(weeks=lag))
    region_frames=[];city_frames=[]
    for region in regions:
        inc=PANELS[region].copy()
        threshold=inc.loc[inc.index<=reference_end].quantile(q).clip(lower=floor)
        if threshold.isna().any(): raise ValueError(f'Cannot define reference thresholds: {region}')
        for c,t in threshold.items():
            THRESHOLD_LOG.append(dict(fold=fold['name'],scenario=scenario,region=region,geocode=c,
                                      reference_end=str(reference_end.date()),threshold=t))
        ratio=inc/threshold
        surges=inc.gt(threshold).astype(float).where(inc.notna())
        count=surges.sum(axis=1).where(surges.notna().all(axis=1))
        event=count.ge(CFG['k']).astype(float).where(count.notna())
        X=pd.DataFrame(index=inc.index)
        for offset in [0,1,2,3,7,11,25,51]:
            a=np.log1p(ratio.shift(lag+offset))
            for stat in ['mean','std','max','min']:
                X[f'r_l{offset}_{stat}']=getattr(a,stat)(axis=1)
            X[f'r_l{offset}_surging']=surges.shift(lag+offset).sum(axis=1,min_count=10)
        X['r_trend']=X['r_l0_mean']-X['r_l3_mean']
        target_dates=inc.index+pd.Timedelta(weeks=h)
        phase=target_dates.dayofyear.to_numpy()/365.2425
        X['season_sin']=np.sin(2*np.pi*phase);X['season_cos']=np.cos(2*np.pi*phase)
        # No absolute year or region/city identifier is used as a predictor.
        valid_history=inc.shift(lag).rolling(4,min_periods=4).count().ge(4).all(axis=1)
        valid_history.iloc[:lag+4]=False
        future_count=count.shift(-h)
        valid=valid_history&future_count.notna()
        xf=X.loc[valid].copy()
        target=target_dates[valid.to_numpy()]
        origin=xf.index
        split=np.repeat('purged',len(xf)).astype(object)
        split[target<=reference_end]='train'
        a=(origin>=pd.Timestamp(fold['cal_a_start']))&(target<=pd.Timestamp(fold['cal_a_end']))&(
            target+pd.Timedelta(weeks=lag)<=pd.Timestamp(fold['cal_b_start']))
        b=(origin>=pd.Timestamp(fold['cal_b_start']))&(target<=pd.Timestamp(fold['cal_b_end']))&(
            target+pd.Timedelta(weeks=lag)<=pd.Timestamp(fold['test_start']))
        test=(origin>=pd.Timestamp(fold['test_start']))&(target<=pd.Timestamp(fold['test_end']))
        split[a]='cal_a';split[b]='cal_b';split[test]='test'
        r=xf.reset_index(names='origin_date')
        r['target_date']=target;r['region']=region;r['split']=split
        r['row_key']=region+'|'+r.origin_date.dt.strftime('%Y-%m-%d')
        r['n']=future_count.loc[valid].astype(int).to_numpy();r['y']=(r.n>=CFG['k']).astype(int)
        region_frames.append(r)
        for city in inc:
            own=pd.DataFrame(index=inc.index)
            for offset in [0,1,2,3,7,11,25,51]:
                own[f'own_l{offset}']=np.log1p(ratio[city]).shift(lag+offset)
            own['own_trend']=own['own_l0']-own['own_l3']
            own['own_mean4']=np.log1p(ratio[city]).shift(lag).rolling(4,min_periods=4).mean()
            c=pd.concat([own.loc[valid].reset_index(drop=True),r.reset_index(drop=True)],axis=1)
            c['geocode']=city;c['city_y']=surges[city].shift(-h).loc[valid].astype(int).to_numpy()
            city_frames.append(c)
        for part in ['train','cal_a','cal_b','test']:
            actual=event.shift(-h).loc[valid].copy()
            actual=actual.iloc[np.flatnonzero(split==part)]
            EVENT_SUPPORT.append(dict(fold=fold['name'],scenario=scenario,horizon=h,region=region,
                split=part,weeks=len(actual),events=int(actual.eq(1).sum()),episodes=episodes_in(actual)))
    return pd.concat(region_frames,ignore_index=True),pd.concat(city_frames,ignore_index=True),scenario

def feature_columns(frame,city=False):
    return [c for c in frame if c.startswith(('r_','season_')) or (city and c.startswith('own_'))]

def check_chronology(regional,fold,lag):
    for stage,next_start in [('train','cal_a_start'),('cal_a','cal_b_start'),('cal_b','test_start')]:
        subset=regional[regional.split.eq(stage)]
        if len(subset):
            assert (subset.target_date+pd.Timedelta(weeks=lag)<=pd.Timestamp(fold[next_start])).all()
    assert not any(c in feature_columns(regional) for c in ['n','y','target_date'])

print('Feature and chronology functions ready. No final-region outcomes have been requested yet.')


## 5. Models and probability distributions
* Logistic regression and histogram gradient boosting predict city surge probabilities.
* Pooled Platt calibration uses development block A only; positive and negative support are required.
* Independent aggregation uses the exact Poisson-binomial count distribution.
* Shared-factor aggregation uses a Gaussian latent factor with one regularized dependence parameter, estimated from full city outcome patterns in later development block B. Gauss–Hermite integration gives the count distribution. It permits positive, exchangeable latent dependence only; it cannot represent arbitrary negative or pair-specific dependence.
* Direct regional logistic regression predicts the binary event, with calibration A when supported.
* Direct proportional-odds ordinal logistic regression predicts counts 0–10 with ordered cutpoints and ridge regularization. Nonconvergence is explicitly unavailable, not silently replaced.
* Seasonal empirical count distributions, a historical distribution and persistence are baselines.

No unavailable calibration is represented by unchanged scores labeled calibrated. The ordinal model's proportional-odds assumption and sparse upper counts remain limitations; external diagnostics are required.

In [ ]:
def binary_pipeline(kind):
    estimator=(LogisticRegression(C=.1,max_iter=3000,random_state=CFG['seed']) if kind=='lr'
      else HistGradientBoostingClassifier(max_iter=CFG['max_model_iterations'],max_leaf_nodes=7,
          min_samples_leaf=30,learning_rate=.05,l2_regularization=5,
          early_stopping=False,random_state=CFG['seed']))
    steps=[SimpleImputer(strategy='median')]
    if kind=='lr': steps.append(StandardScaler())
    return make_pipeline(*steps,estimator)

class BinaryModel:
    def __init__(self,kind):self.kind=kind
    def fit(self,X,y):
        self.prior=(np.sum(y)+1)/(len(y)+2)
        self.model=binary_pipeline(self.kind).fit(X,y) if np.unique(y).size==2 else None
        return self
    def predict(self,X):
        return np.repeat(self.prior,len(X)) if self.model is None else self.model.predict_proba(X)[:,1]

def logit(p):
    p=np.clip(np.asarray(p),1e-6,1-1e-6)
    return np.log(p/(1-p)).reshape(-1,1)

class Platt:
    def fit(self,p,y):
        y=np.asarray(y);self.n_pos=int(y.sum());self.n_neg=int(len(y)-y.sum())
        self.available=self.n_pos>=CFG['min_cal_positives'] and self.n_neg>=CFG['min_cal_negatives']
        self.reason='available' if self.available else 'insufficient positive/negative calibration support'
        self.model=LogisticRegression(C=1,max_iter=2000).fit(logit(p),y) if self.available else None
        return self
    def predict(self,p):
        if not self.available: raise RuntimeError('Calibration unavailable; never label unchanged scores as calibrated')
        return self.model.predict_proba(logit(p))[:,1]

def poisson_binomial(p):
    p=np.asarray(p,dtype=float)
    pmf=np.zeros((len(p),p.shape[1]+1));pmf[:,0]=1
    for j in range(p.shape[1]):
        old=pmf.copy();pmf=old*(1-p[:,j,None]);pmf[:,1:]+=old[:,:-1]*p[:,j,None]
    return pmf/pmf.sum(axis=1,keepdims=True)

H_NODES,H_WEIGHTS=hermgauss(CFG['quadrature_nodes'])
Z_NODES=H_NODES*np.sqrt(2);H_WEIGHTS=H_WEIGHTS/np.sqrt(np.pi)
def conditional_city_probabilities(p,rho,z):
    return np.clip(ndtr((ndtri(np.clip(p,1e-6,1-1e-6))-np.sqrt(rho)*z)/np.sqrt(1-rho)),1e-9,1-1e-9)

def factor_count_distribution(p,rho):
    if rho<1e-10:return poisson_binomial(p)
    result=np.zeros((len(p),p.shape[1]+1))
    for z,w in zip(Z_NODES,H_WEIGHTS):
        result+=w*poisson_binomial(conditional_city_probabilities(p,rho,z))
    return result/result.sum(axis=1,keepdims=True)

def fit_dependence(p,y):
    # Gaussian shared-factor Bernoulli model; positive dependence only.
    # Marginal probabilities are identical to the independent comparator.
    if len(p)<CFG['min_dependence_weeks'] or y.sum()<20 or (1-y).sum()<20:
        return None,dict(available=False,reason='insufficient full-pattern support',n=len(p))
    def loss(rho):
        logs=[]
        for z,w in zip(Z_NODES,H_WEIGHTS):
            cp=conditional_city_probabilities(p,rho,z)
            logs.append(np.log(w)+(y*np.log(cp)+(1-y)*np.log1p(-cp)).sum(axis=1))
        return -logsumexp(np.stack(logs,axis=1),axis=1).mean()+CFG['rho_penalty']*rho*rho
    result=minimize_scalar(loss,bounds=(0,CFG['rho_max']),method='bounded',options={'xatol':1e-4})
    if not result.success: return None,dict(available=False,reason='dependence optimizer failure',n=len(p))
    rho=float(result.x) if result.fun<loss(0) else 0.
    return rho,dict(available=True,reason='available',n=len(p),rho=rho,
                    boundary=bool(rho==0 or rho>CFG['rho_max']-.01),objective=float(loss(rho)))

class OrdinalLogit:
    # Proportional-odds count model: ordered cutpoints and one common coefficient vector.
    # Fits classes 0..10 even if some training count classes are absent.
    def fit(self,X,y):
        self.pre=make_pipeline(SimpleImputer(strategy='median'),StandardScaler())
        xx=self.pre.fit_transform(X);y=np.asarray(y,dtype=int);dim=xx.shape[1]
        cum=(np.array([(y<=k).sum() for k in range(10)])+.5)/(len(y)+1)
        cuts=np.log(cum/(1-cum))
        cuts=np.maximum.accumulate(cuts+np.arange(10)*1e-3)
        initial=np.r_[np.zeros(dim),cuts[0],np.log(np.expm1(np.maximum(np.diff(cuts),1e-3)))]
        def unpack(v):
            raw=v[dim:];ct=np.r_[raw[0],raw[0]+np.cumsum(np.logaddexp(0,raw[1:]))]
            return v[:dim],ct,raw
        def objective(v):
            beta,ct,raw=unpack(v)
            F=expit(ct[None,:]-xx@beta[:,None]);f=F*(1-F)
            ff=np.column_stack([np.zeros(len(y)),F,np.ones(len(y))])
            pp=np.maximum(ff[np.arange(len(y)),y+1]-ff[np.arange(len(y)),y],1e-12)
            fpad=np.column_stack([np.zeros(len(y)),f,np.zeros(len(y))])
            geta=(fpad[np.arange(len(y)),y+1]-fpad[np.arange(len(y)),y])/pp
            gb=xx.T@geta/len(y)+.01*beta
            gc=np.array([((-((y==k).astype(float))+(y==k+1).astype(float))*f[:,k]/pp).mean() for k in range(10)])
            gr=np.r_[gc.sum(),np.cumsum(gc[::-1])[::-1][1:]*expit(raw[1:])]
            return -np.log(pp).mean()+.005*np.sum(beta*beta),np.r_[gb,gr]
        self.result=minimize(objective,initial,jac=True,method='L-BFGS-B',options={'maxiter':800,'ftol':1e-9})
        self.beta,self.cuts,_=unpack(self.result.x)
        self.available=bool(self.result.success)
        return self
    def predict(self,X):
        if not self.available:raise RuntimeError('Ordinal optimization failed; output unavailable')
        F=expit(self.cuts[None,:]-self.pre.transform(X)@self.beta[:,None])
        pmf=np.diff(np.column_stack([np.zeros(len(F)),F,np.ones(len(F))]),axis=1)
        return np.maximum(pmf,0)/np.maximum(pmf.sum(axis=1,keepdims=True),1e-12)

def seasonal_distribution(train,test):
    train_week=train.target_date.dt.isocalendar().week.to_numpy()
    test_week=test.target_date.dt.isocalendar().week.to_numpy()
    distributions=[]
    for w in test_week:
        dist=np.abs(train_week-w);near=np.minimum(dist,53-dist)<=4
        counts=np.bincount(train.loc[near,'n'].to_numpy(),minlength=11)+.5
        distributions.append(counts/counts.sum())
    return np.asarray(distributions)

print('Models ready: logistic, gradient boosting, ordinal logistic, and a regularized shared-factor model.')


## 6. Out-of-time development experiments
Two expanding-history development evaluations target 2021 and 2022. The final fixed model uses historical fitting targets through 2018, marginal calibration in 2019–2020, dependence estimation in 2021–2022 and evaluation in 2023–2025, with availability purging at all boundaries. Models pool development regions using transferable features, not region identifiers. Settings are prespecified and not selected from these development rankings.

In [ ]:
FORECASTS=[]; CITY_FORECASTS=[]; STATUS=[]; AUDIT=[]; FITTED={}
def city_matrix(city,p,regional):
    work=city[['row_key','region','geocode','city_y']].copy();work['p']=p
    groups={key:g for key,g in work.groupby('row_key',sort=False)}
    pp=[];yy=[]
    for key,region in regional[['row_key','region']].itertuples(index=False,name=None):
        g=groups[key].set_index('geocode').reindex(list(REGIONS[region]['cities']))
        assert len(g)==10 and g[['p','city_y']].notna().all().all()
        pp.append(g.p.to_numpy());yy.append(g.city_y.to_numpy())
    return np.asarray(pp),np.asarray(yy)

def record_forecast(regional,method,pmf=None,prob=None,**tags):
    result=regional[['region','row_key','origin_date','target_date','n','y']].copy()
    result['method']=method
    if pmf is not None:
        assert np.isfinite(pmf).all() and np.allclose(pmf.sum(axis=1),1,atol=1e-7)
        assert (pmf>=-1e-10).all()
        for i in range(11):result[f'p{i}']=pmf[:,i]
        result['prob_joint']=pmf[:,CFG['k']:].sum(axis=1)
        result['expected_n']=pmf@np.arange(11)
    else:
        result['prob_joint']=np.asarray(prob)
        result['expected_n']=np.nan
        for i in range(11):result[f'p{i}']=np.nan
    for key,value in tags.items():result[key]=value
    result['evaluation']=np.where(result.region.isin(EXT),'external','development_temporal')
    FORECASTS.append(result)

def run_experiment(fold,h,regions,lag=None,q=None,floor=None):
    lag=CFG['reporting_lag'] if lag is None else lag
    regional,city,scenario=make_dataset(regions,fold,h,lag,q,floor)
    check_chronology(regional,fold,lag)
    rcols=feature_columns(regional);ccols=feature_columns(city,city=True)
    tr=regional.split.eq('train')&regional.region.isin(DEV)
    aa=regional.split.eq('cal_a')&regional.region.isin(DEV)
    bb=regional.split.eq('cal_b')&regional.region.isin(DEV)
    test=regional.split.eq('test')
    ct=city.split.eq('train')&city.region.isin(DEV)
    ca=city.split.eq('cal_a')&city.region.isin(DEV)
    if tr.sum()<100 or ct.sum()<1000 or test.sum()<20:
        raise RuntimeError('Insufficient rows for fixed experiment; do not reinterpret this as a negative scientific result.')
    rt=regional.loc[test].copy();rb=regional.loc[bb].copy()
    tags=dict(fold=fold['name'],scenario=scenario,horizon=h,lag=lag,
              quantile=CFG['quantile'] if q is None else q,floor=CFG['incidence_floor'] if floor is None else floor)
    AUDIT.append(dict(**tags,train_region_weeks=int(tr.sum()),train_city_weeks=int(ct.sum()),
        cal_a_region_weeks=int(aa.sum()),cal_a_joint_events=int(regional.loc[aa,'y'].sum()),
        cal_b_region_weeks=int(bb.sum()),cal_b_joint_events=int(regional.loc[bb,'y'].sum()),
        max_training_target=str(regional.loc[tr,'target_date'].max()),
        min_calibration_origin=str(regional.loc[aa,'origin_date'].min()),
        min_test_origin=str(rt.origin_date.min())))
    # Baselines all return a full count distribution.
    prior=np.bincount(regional.loc[tr,'n'].to_numpy(),minlength=11)+.5;prior=prior/prior.sum()
    record_forecast(rt,'historical_distribution',pmf=np.tile(prior,(len(rt),1)),**tags)
    record_forecast(rt,'seasonal_distribution',pmf=seasonal_distribution(regional.loc[tr],rt),**tags)
    persist=np.eye(11)[rt.r_l0_surging.astype(int).to_numpy()]
    record_forecast(rt,'persistence',pmf=persist,**tags)
    for family in ['lr','gb']:
        model=BinaryModel(family).fit(city.loc[ct,ccols],city.loc[ct,'city_y'])
        rawp=model.predict(city[ccols])
        cal=Platt().fit(rawp[ca],city.loc[ca,'city_y'])
        prefix=family+('_cal' if cal.available else '_raw')
        p=cal.predict(rawp) if cal.available else rawp
        STATUS.append(dict(**tags,component=family+'_city_calibration',available=cal.available,
          positives=cal.n_pos,negatives=cal.n_neg,reason=cal.reason))
        full_p,full_y=city_matrix(city,p,regional)
        assert np.array_equal(full_y.sum(axis=1),regional.n.to_numpy()),'City/regional target alignment failed'
        ptest=full_p[test.to_numpy()];pb=full_p[bb.to_numpy()];yb=full_y[bb.to_numpy()]
        indep=poisson_binomial(ptest)
        record_forecast(rt,prefix+'_independent',pmf=indep,**tags)
        rho,depstatus=fit_dependence(pb,yb)
        STATUS.append(dict(**tags,component=family+'_dependence',**depstatus))
        if rho is not None:
            dependent=factor_count_distribution(ptest,rho)
            # The two aggregations must preserve the SAME city marginals (quadrature tolerance).
            assert np.allclose(indep@np.arange(11),dependent@np.arange(11),atol=2e-3)
            record_forecast(rt,prefix+'_shared_factor',pmf=dependent,**tags)
        cp=city.loc[city.split.eq('test'),['region','geocode','origin_date','target_date','city_y']].copy()
        cp['probability']=p[city.split.eq('test')];cp['family']=family;cp['calibrated']=cal.available
        for key,value in tags.items():cp[key]=value
        CITY_FORECASTS.append(cp)
        FITTED[(fold['name'],scenario,h,family)]=dict(model=model,calibrator=cal,rho=rho,columns=ccols)
    direct=BinaryModel('lr').fit(regional.loc[tr,rcols],regional.loc[tr,'y'])
    directp=direct.predict(regional[rcols])
    record_forecast(rt,'direct_regional_lr_raw',prob=directp[test],**tags)
    dcal=Platt().fit(directp[aa],regional.loc[aa,'y'])
    STATUS.append(dict(**tags,component='direct_lr_calibration',available=dcal.available,
                       positives=dcal.n_pos,negatives=dcal.n_neg,reason=dcal.reason))
    if dcal.available:record_forecast(rt,'direct_regional_lr_cal',prob=dcal.predict(directp[test]),**tags)
    ordinal=OrdinalLogit().fit(regional.loc[tr,rcols],regional.loc[tr,'n'])
    STATUS.append(dict(**tags,component='ordinal_fit',available=ordinal.available,
                       reason=str(ordinal.result.message),iterations=int(ordinal.result.nit)))
    if ordinal.available:record_forecast(rt,'direct_ordinal_lr',pmf=ordinal.predict(rt[rcols]),**tags)
    print('Completed',fold['name'],scenario,'h=',h,'test region-weeks=',len(rt),flush=True)

# Fixed parameters, not an unreported tuning sweep. All methods have zero search trials.
for fold in FOLDS[:2]:
    for h in CFG['horizons']:run_experiment(fold,h,DEV)
pd.DataFrame(STATUS).to_csv(TABLES/'development_model_status.csv',index=False)
pd.DataFrame(EVENT_SUPPORT).to_csv(TABLES/'development_event_support.csv',index=False)
print('Development evaluation complete. Final methods remain the prespecified ones, regardless of rankings.')


## 7. Development support and precision diagnostic
Before fetching external regions, verify support for the primary calibrated city probabilities and dependence estimation. The diagnostic bootstrap of out-of-time development RPS differences provides a preliminary sense of precision; it is **not a formal power calculation or a guarantee of sufficient external episodes**. External panel size is not adjusted to chase a favorable result. If support fails, export the diagnostic ZIP and explicitly revise the protocol before consuming external evaluation.

In [ ]:
# Development-only support check BEFORE requesting external-region outcomes.
pre_r,pre_c,pre_s=make_dataset(DEV,FOLDS[-1],CFG['primary_horizon'])
pre_a=pre_c[pre_c.split.eq('cal_a')]
pre_b=pre_r[pre_r.split.eq('cal_b')]
pre_bc=pre_c[pre_c.split.eq('cal_b')]
preflight=dict(city_calibration_positives=int(pre_a.city_y.sum()),
 city_calibration_negatives=int(len(pre_a)-pre_a.city_y.sum()),
 dependence_region_weeks=len(pre_b),dependence_positive_city_weeks=int(pre_bc.city_y.sum()),
 dependence_negative_city_weeks=int(len(pre_bc)-pre_bc.city_y.sum()),
 training_region_weeks=int(pre_r.split.eq('train').sum()),
 uses_external_outcomes=False)
preflight['basic_primary_support_ok']=bool(
 preflight['city_calibration_positives']>=CFG['min_cal_positives'] and
 preflight['city_calibration_negatives']>=CFG['min_cal_negatives'] and
 preflight['dependence_region_weeks']>=CFG['min_dependence_weeks'] and
 preflight['dependence_positive_city_weeks']>=20 and
 preflight['dependence_negative_city_weeks']>=20)
(OUT/'development_preflight.json').write_text(json.dumps(preflight,indent=2))
print('Development-only preflight:',preflight)

# Exploratory precision diagnostic using out-of-time development losses, not a power guarantee.
dev_predictions=pd.concat(FORECASTS,ignore_index=True)
dev=dev_predictions[(dev_predictions.horizon==2)&(dev_predictions.scenario==pre_s)]
losses=[]
for candidate,baseline in [('lr_cal_shared_factor','lr_cal_independent')]:
    a=dev[dev.method.eq(candidate)].copy();b=dev[dev.method.eq(baseline)].copy()
    if len(a) and len(b):
        def pre_rps(g):
            pp=g[[f'p{i}' for i in range(11)]].to_numpy()
            return ((pp.cumsum(axis=1)[:,:10]-(g.n.to_numpy()[:,None]<=np.arange(10)))**2).mean(axis=1)
        a['loss']=pre_rps(a);b['loss']=pre_rps(b)
        paired=a[['row_key','fold','region','target_date','loss']].merge(
          b[['row_key','fold','loss']],on=['row_key','fold'],suffixes=('_a','_b'),validate='one_to_one')
        paired['difference']=paired.loss_a-paired.loss_b
        matrix=paired.pivot(index='target_date',columns='region',values='difference')
        values=matrix.to_numpy();rng_pre=np.random.default_rng(CFG['seed']);draws=[]
        block=min(CFG['block_weeks'],len(values))
        for _ in range(500):
            starts=rng_pre.integers(0,len(values)-block+1,size=int(np.ceil(len(values)/block)))
            ii=np.concatenate([np.arange(s,s+block) for s in starts])[:len(values)]
            draws.append(np.nanmean(values[ii],axis=0).mean())
        lo,hi=np.nanquantile(draws,[.025,.975])
        losses.append(dict(comparison='matched calibrated logistic dependence',
          development_difference=float(matrix.mean().mean()),ci_low=float(lo),ci_high=float(hi),
          diagnostic_only=True,formal_external_sample_size_guarantee=False))
pd.DataFrame(losses).to_csv(TABLES/'development_precision_diagnostic.csv',index=False)
if not preflight['basic_primary_support_ok']:
    raise RuntimeError('Primary development support gate failed. External data were not requested. Export diagnostics and redesign explicitly; do not manufacture a calibration result.')


## 8. Fixed final evaluation and prespecified sensitivity grid
This cell downloads the two external panels if enabled, then runs the locked models. Final 2023–2025 results for development panels are temporal assessments (Rio is already pilot-seen); Fortaleza and Recife supply geographic evaluation. Sensitivity scenarios vary lag (1/4 weeks), threshold quantile (.85/.95) and incidence floor (5/20), one change at a time at the primary horizon. K=2/4 sensitivity is derived from the same predictive count distributions later.

All scenarios are retained. Scores across different outcome definitions are not directly comparable measures of improvement; do not select the most favorable definition. Calibration blocks and reference cutoffs are repurged for each lag. Longer lags do not recreate historical data vintages.

In [ ]:
# The protocol hash was written before any source download.
# Running this cell consumes the external evaluation. Do not modify the method after inspecting it
# and then describe the same panels as untouched confirmation.
if CFG['run_external']:
    source_ext=download_regions(EXT)
    source_ext.to_csv(OUT/'source_external.csv',index=False)
    quality_all=build_panels(source_ext,EXT)
    evaluation_regions=DEV+EXT
else:
    evaluation_regions=DEV
    print('External evaluation disabled. Output will not establish geographic generalization.')
for h in CFG['horizons']:run_experiment(FOLDS[-1],h,evaluation_regions)
if CFG['run_sensitivity']:
    # Prespecified secondary analyses. All are retained; none selects the main result.
    for lag,q,floor in [(1,.90,10.),(4,.90,10.),(2,.85,10.),(2,.95,10.),(2,.90,5.),(2,.90,20.)]:
        run_experiment(FOLDS[-1],CFG['primary_horizon'],evaluation_regions,lag=lag,q=q,floor=floor)
predictions=pd.concat(FORECASTS,ignore_index=True)
city_predictions=pd.concat(CITY_FORECASTS,ignore_index=True)
predictions.to_csv(TABLES/'all_regional_forecasts.csv',index=False)
city_predictions.to_csv(TABLES/'all_city_forecasts.csv',index=False)
status=pd.DataFrame(STATUS);status.to_csv(TABLES/'model_status.csv',index=False)
pd.DataFrame(AUDIT).to_csv(TABLES/'chronology_audit.csv',index=False)
pd.DataFrame(THRESHOLD_LOG).drop_duplicates().to_csv(TABLES/'surge_thresholds.csv',index=False)
event_support=pd.DataFrame(EVENT_SUPPORT).drop_duplicates()
event_support.to_csv(TABLES/'event_support.csv',index=False)
MAIN=f'q{CFG["quantile"]:g}_floor{CFG["incidence_floor"]:g}_lag{CFG["reporting_lag"]}'
display(event_support[(event_support.fold=='final')&(event_support.scenario==MAIN)&(event_support.horizon==2)])


## 9. Forecast performance, count distributions, and calibration
Count RPS is normalized by 10 and scored over cumulative probabilities for counts 0–9. The binary Brier and log scores assess ≥3 surges; average precision and AUROC assess ranking. Ninety-percent discrete prediction interval coverage and width are reported. Performance is first calculated by region, then averaged with **equal region weights**; pooled city rows are not independent replication. Calibration bins and year-specific results are exported. Binary-only models have no count RPS and are not used to infer demand distributions.

In [ ]:
PMF_COLS=[f'p{i}' for i in range(11)]
def metric_row(g):
    y=g.y.to_numpy();p=np.clip(g.prob_joint.to_numpy(),1e-9,1-1e-9)
    row=dict(n_weeks=len(g),event_weeks=int(y.sum()),event_rate=float(y.mean()),
      brier=float(np.mean((p-y)**2)),binary_log_score=float(-(y*np.log(p)+(1-y)*np.log1p(-p)).mean()),
      auroc=float(roc_auc_score(y,p)) if np.unique(y).size==2 else np.nan,
      average_precision=float(average_precision_score(y,p)) if y.sum()>0 else np.nan)
    pmf=g[PMF_COLS].to_numpy()
    if np.isfinite(pmf).all():
        cdf=pmf.cumsum(axis=1)[:,:10]
        truth=(g.n.to_numpy()[:,None]<=np.arange(10)).astype(float)
        row['rps']=float(np.mean(np.mean((cdf-truth)**2,axis=1)))
        row['count_log_score']=float(-np.log(np.maximum(pmf[np.arange(len(g)),g.n.to_numpy()],1e-12)).mean())
        low=(pmf.cumsum(axis=1)>=.05).argmax(axis=1);high=(pmf.cumsum(axis=1)>=.95).argmax(axis=1)
        row['interval90_coverage']=float(((g.n.to_numpy()>=low)&(g.n.to_numpy()<=high)).mean())
        row['interval90_width']=float((high-low).mean())
        row['count_mae']=float(np.mean(abs(g.expected_n-g.n)))
    return row

GROUP=['fold','scenario','horizon','evaluation','region','method']
metric_records=[];year_records=[];bin_records=[];k_records=[]
for key,g in predictions.groupby(GROUP,sort=False):
    tags=dict(zip(GROUP,key));metric_records.append(dict(**tags,**metric_row(g)))
    for year,gg in g.groupby(g.target_date.dt.year):
        year_records.append(dict(**tags,year=int(year),**metric_row(gg)))
    bins=pd.cut(g.prob_joint,bins=np.linspace(0,1,11),include_lowest=True)
    for bin_id,gg in g.groupby(bins,observed=True):
        bin_records.append(dict(**tags,bin=str(bin_id),n=len(gg),
          predicted=gg.prob_joint.mean(),observed=gg.y.mean()))
    if g[PMF_COLS].notna().all().all():
        for k in [2,3,4]:
            p=g[PMF_COLS[k:]].sum(axis=1).to_numpy();y=g.n.ge(k).to_numpy()
            k_records.append(dict(**tags,k=k,n=len(g),events=int(y.sum()),brier=float(np.mean((p-y)**2))))
metrics=pd.DataFrame(metric_records);metrics.to_csv(TABLES/'metrics_by_region.csv',index=False)
year_metrics=pd.DataFrame(year_records);year_metrics.to_csv(TABLES/'metrics_by_year.csv',index=False)
pd.DataFrame(bin_records).to_csv(TABLES/'calibration_bins.csv',index=False)
pd.DataFrame(k_records).to_csv(TABLES/'k_sensitivity.csv',index=False)
score_columns=[c for c in ['brier','binary_log_score','auroc','average_precision','rps',
 'count_log_score','interval90_coverage','interval90_width','count_mae'] if c in metrics]
summary=metrics.groupby(['fold','scenario','horizon','evaluation','method'])[score_columns].mean().reset_index()
summary.to_csv(TABLES/'metrics_equal_region_summary.csv',index=False)
display(summary[(summary.fold=='final')&(summary.scenario==MAIN)&summary.horizon.eq(2)].sort_values(['evaluation','rps']))

# City-level calibration/ranking is scored separately: better dependence should not be confused
# with changing the city-level prediction model.
city_scores=[]
for key,g in city_predictions.groupby(['fold','scenario','horizon','family','calibrated','region']):
    tags=dict(zip(['fold','scenario','horizon','family','calibrated','region'],key))
    p=np.clip(g.probability.to_numpy(),1e-9,1-1e-9);y=g.city_y.to_numpy()
    city_scores.append(dict(**tags,n=len(g),positives=int(y.sum()),brier=float(np.mean((p-y)**2)),
      auroc=roc_auc_score(y,p) if np.unique(y).size==2 else np.nan))
pd.DataFrame(city_scores).to_csv(TABLES/'city_probability_metrics.csv',index=False)


## 10. Paired uncertainty and robustness
Calendar blocks of 13 weeks are resampled jointly across regions (1,000 replicates), retaining cross-region contemporaneous dependence. Intervals condition on the selected panels and fixed fitted models; they do not include every source of model-fitting, data-revision or region-selection uncertainty. With two external regions, they cannot establish generalization to all regions.

Primary inference is the matched calibrated logistic comparison for external 2-week count RPS. Other comparisons are exploratory secondary analyses, not a multiplicity-adjusted search for significance. Region and episode omission tables assess concentration of gains; these omit evaluation scores without refitting and are not new validation datasets. Discrete episode fragmentation remains sensitive to the operational surge threshold.

In [ ]:
def row_rps(frame):
    cdf=frame[PMF_COLS].to_numpy().cumsum(axis=1)[:,:10]
    truth=(frame.n.to_numpy()[:,None]<=np.arange(10)).astype(float)
    return ((cdf-truth)**2).mean(axis=1)

def block_sample(n,block,rng):
    block=min(block,n)
    starts=rng.integers(0,n-block+1,size=int(np.ceil(n/block)))
    return np.concatenate([np.arange(s,s+block) for s in starts])[:n]

PAIRS=[('lr_cal_shared_factor','lr_cal_independent','primary_matched_dependence'),
       ('gb_cal_shared_factor','gb_cal_independent','secondary_matched_dependence'),
       ('lr_raw_shared_factor','lr_raw_independent','exploratory_uncalibrated'),
       ('gb_raw_shared_factor','gb_raw_independent','exploratory_uncalibrated'),
       ('direct_ordinal_lr','seasonal_distribution','secondary_direct_count')]
ci_records=[];leave_region=[];leave_episode=[]
rng=np.random.default_rng(CFG['seed'])
final=predictions[(predictions.fold=='final')&(predictions.scenario==MAIN)]
for (h,evaluation),group in final.groupby(['horizon','evaluation']):
    for candidate,baseline,kind in PAIRS:
        a=group[group.method.eq(candidate)].copy();b=group[group.method.eq(baseline)].copy()
        if a.empty or b.empty:
            ci_records.append(dict(horizon=h,evaluation=evaluation,candidate=candidate,baseline=baseline,
              comparison=kind,available=False,reason='Required fitted/calibrated method unavailable'))
            continue
        a['rps_loss']=row_rps(a);a['brier_loss']=(a.prob_joint-a.y)**2
        b['rps_loss']=row_rps(b);b['brier_loss']=(b.prob_joint-b.y)**2
        joined=a[['row_key','region','target_date','n','y','rps_loss','brier_loss']].merge(
          b[['row_key','rps_loss','brier_loss']],on='row_key',suffixes=('_candidate','_baseline'),validate='one_to_one')
        for score in ['rps','brier']:
            joined['difference']=joined[score+'_loss_candidate']-joined[score+'_loss_baseline']
            # Resample calendar blocks together across ALL regions, retaining contemporaneous dependence.
            delta=joined.pivot(index='target_date',columns='region',values='difference')
            grid=pd.date_range(delta.index.min(),delta.index.max(),freq='W-SUN');delta=delta.reindex(grid)
            values=delta.to_numpy();draws=[]
            for _ in range(CFG['bootstrap_reps']):
                sample=values[block_sample(len(values),CFG['block_weeks'],rng)]
                region_mean=np.nanmean(sample,axis=0)
                draws.append(np.mean(region_mean))
            lo,hi=np.nanquantile(draws,[.025,.975])
            point=float(delta.mean().mean())
            ci_records.append(dict(horizon=h,evaluation=evaluation,candidate=candidate,baseline=baseline,
              comparison=kind,score=score,available=True,regions=len(delta.columns),weeks=len(delta),
              difference=point,ci_low=lo,ci_high=hi,
              reason='Negative favors candidate; time-block CI conditional on selected regions; fixed fitted models'))
            for region in delta:
                if len(delta.columns)>1:
                    leave_region.append(dict(horizon=h,evaluation=evaluation,candidate=candidate,
                      baseline=baseline,score=score,omitted_region=region,
                      difference=float(delta.drop(columns=region).mean().mean())))
            # Omit each observed epidemic episode from the score; this is sensitivity analysis,
            # not refitting or an independent replication.
            for region,gg in joined.groupby('region'):
                gg=gg.sort_values('target_date').copy()
                start=gg.y.eq(1)&(gg.y.shift().ne(1)|gg.target_date.diff().ne(pd.Timedelta(weeks=1)))
                gg['episode']=start.cumsum()
                for eid,ep in gg[gg.y.eq(1)].groupby('episode'):
                    remaining=joined[~joined.row_key.isin(ep.row_key)]
                    leave_episode.append(dict(horizon=h,evaluation=evaluation,candidate=candidate,
                      baseline=baseline,score=score,omitted_region=region,episode=int(eid),
                      episode_onset=str(ep.target_date.min().date()),weeks_removed=len(ep),
                      difference=float(remaining.groupby('region').difference.mean().mean())))
intervals=pd.DataFrame(ci_records);intervals.to_csv(TABLES/'paired_block_bootstrap.csv',index=False)
pd.DataFrame(leave_region).to_csv(TABLES/'leave_one_region_out_scores.csv',index=False)
pd.DataFrame(leave_episode).to_csv(TABLES/'leave_one_episode_out_scores.csv',index=False)
display(intervals[(intervals.evaluation=='external')&intervals.horizon.eq(2)])


## 11. Preparedness decisions — explicitly hypothetical
One surging city is assigned one identical generic service unit. These units are **not observed hospital beds, supplies, patient demand or prevented deaths**. For capacity C and reserve R, predicted shortage reduction is E[min(R, max(N−C,0))]. Release when expected benefit exceeds reserve activation cost.

All policies face identical capacities, reserve amounts and unit costs. Compare (a) unconstrained cost-based releases and (b) online token budgets allowing at most 20% of eligible forecast weeks in each target year. Tokens accrue sequentially, the bank holds at most one release, and there is no hindsight top-k selection. `always_request` under the budget becomes a periodic eligible release comparator. Realized activation can be below the limit and is always reported.

The grid covers C=1/2/4, R=1/2/4 and reserve-unit costs=.1/.5/1/2 relative to one unit of unmet hypothetical demand. The prespecified illustrated scenario is C=2, R=2, cost=.5. Calendar/resource logistics, travel, inventory carryover and replenishment are not modeled. Paired cost intervals are secondary. Episode warnings count only actions whose forecast origin precedes episode onset and whose target falls inside that episode; boundary-left-censored episodes are labeled.

In [ ]:
# These are generic service units, NOT inferred hospital beds, patient demand, or prevented deaths.
DECISIONS=[];DECISION_WEEKLY=[];WARNINGS=[]
def expected_benefit(pmf,C,R):
    return pmf@np.minimum(R,np.maximum(np.arange(11)-C,0))

def online_actions(benefit,R,unit_cost,budget=None):
    request=np.asarray(benefit)>R*unit_cost
    if budget is None:return request.astype(int)
    bank=0.;actions=[]
    for wanted in request:
        bank=min(1.,bank+budget)
        use=bool(wanted and bank>=1-1e-10)
        if use:bank-=1
        actions.append(int(use))
    assert sum(actions)<=np.floor(budget*len(actions)+1e-8)
    return np.asarray(actions)

decision_source=final.copy() # Primary definition, all three horizons, fixed final chronology.
for (h,region),rr in decision_source.groupby(['horizon','region']):
    template=rr[rr.method.eq('seasonal_distribution')].sort_values('target_date')
    methods={name:g.sort_values('target_date') for name,g in rr.groupby('method')
             if g[PMF_COLS].notna().all().all()}
    # Binary-only direct logistic is not a distribution of demand, so it is not used in this simulation.
    for C in [1,2,4]:
        for R in [1,2,4]:
            for cost in [.1,.5,1.,2.]:
                policy_frames=dict(methods)
                policy_frames['never_release']=template;policy_frames['always_request']=template
                for policy,g in policy_frames.items():
                    assert g.row_key.tolist()==template.row_key.tolist()
                    pmf=g[PMF_COLS].to_numpy();benefit=expected_benefit(pmf,C,R)
                    if policy=='never_release':benefit=np.repeat(-np.inf,len(g))
                    if policy=='always_request':benefit=np.repeat(np.inf,len(g))
                    for mode,budget in [('cost_based',None),('online_budget_20pct',.2)]:
                        # Token budgets restart each target year. No top-k selection using future scores.
                        actions=np.zeros(len(g),dtype=int)
                        for year in sorted(g.target_date.dt.year.unique()):
                            ix=np.flatnonzero(g.target_date.dt.year.to_numpy()==year)
                            actions[ix]=online_actions(benefit[ix],R,cost,budget)
                        demand=g.n.to_numpy();short=np.maximum(demand-C-R*actions,0)
                        loss=short+cost*R*actions
                        joint_events=g.y.to_numpy()
                        true_alerts=int(((actions==1)&(joint_events==1)).sum())
                        DECISIONS.append(dict(horizon=h,region=region,evaluation=g.evaluation.iloc[0],
                          policy=policy,mode=mode,capacity=C,reserve=R,reserve_unit_cost=cost,
                          weeks=len(g),activation_rate=actions.mean(),activated_units=float(R*actions.sum()),
                          unmet_units=float(short.sum()),weeks_short=int((short>0).sum()),mean_cost=loss.mean(),
                          joint_event_precision=true_alerts/actions.sum() if actions.sum() else np.nan,
                          joint_event_recall=true_alerts/joint_events.sum() if joint_events.sum() else np.nan,
                          false_alerts=int(((actions==1)&(joint_events==0)).sum()),
                          missed_event_weeks=int(((actions==0)&(joint_events==1)).sum())))
                        if C==2 and R==2 and cost==.5:
                            w=g[['region','row_key','origin_date','target_date','n','y','evaluation']].copy()
                            w['horizon']=h;w['policy']=policy;w['mode']=mode
                            w['action']=actions;w['unmet_units']=short;w['cost']=loss
                            DECISION_WEEKLY.append(w)
    # Episode warnings use the explicitly budget-constrained primary reserve scenario.
for g in DECISION_WEEKLY:
    if g['mode'].iloc[0]!='online_budget_20pct':continue
    gg=g.sort_values('target_date').copy()
    starts=gg.y.eq(1)&(gg.y.shift().ne(1)|gg.target_date.diff().ne(pd.Timedelta(weeks=1)))
    gg['episode']=starts.cumsum()
    for eid,ep in gg[gg.y.eq(1)].groupby('episode'):
        onset=ep.target_date.min()
        hits=ep[(ep.action==1)&(ep.origin_date<onset)]
        WARNINGS.append(dict(region=gg.region.iloc[0],horizon=int(gg.horizon.iloc[0]),
          policy=gg.policy.iloc[0],evaluation=gg.evaluation.iloc[0],episode=int(eid),
          onset=str(onset.date()),duration_weeks=len(ep),warned=bool(len(hits)),
          lead_weeks=(onset-hits.origin_date.min()).days/7 if len(hits) else np.nan,
          left_censored=bool(onset==gg.target_date.min())))
decisions=pd.DataFrame(DECISIONS);decisions.to_csv(TABLES/'reserve_scenario_grid.csv',index=False)
decision_weekly=pd.concat(DECISION_WEEKLY,ignore_index=True)
decision_weekly.to_csv(TABLES/'reserve_weekly_primary_scenario.csv',index=False)
pd.DataFrame(WARNINGS).to_csv(TABLES/'episode_warnings_budget_constrained.csv',index=False)
cost_intervals=[]
for (h,mode,evaluation),g in decision_weekly.groupby(['horizon','mode','evaluation']):
    aa=g[g.policy.eq('lr_cal_shared_factor')][['row_key','region','target_date','cost']]
    bb=g[g.policy.eq('lr_cal_independent')][['row_key','cost']]
    if aa.empty or bb.empty:continue
    paired=aa.merge(bb,on='row_key',suffixes=('_dependent','_independent'),validate='one_to_one')
    paired['difference']=paired.cost_dependent-paired.cost_independent
    pivot=paired.pivot(index='target_date',columns='region',values='difference')
    pivot=pivot.reindex(pd.date_range(pivot.index.min(),pivot.index.max(),freq='W-SUN'))
    values=pivot.to_numpy();draws=[]
    for _ in range(CFG['bootstrap_reps']):
        sample=values[block_sample(len(values),CFG['block_weeks'],rng)]
        draws.append(np.nanmean(sample,axis=0).mean())
    lo,hi=np.nanquantile(draws,[.025,.975])
    cost_intervals.append(dict(horizon=h,mode=mode,evaluation=evaluation,capacity=2,reserve=2,
       reserve_unit_cost=.5,difference=pivot.mean().mean(),ci_low=lo,ci_high=hi,
       meaning='Dependent minus independent hypothetical mean cost; secondary endpoint'))
pd.DataFrame(cost_intervals).to_csv(TABLES/'paired_simulation_cost_intervals.csv',index=False)
display(decisions[(decisions.horizon==2)&(decisions.evaluation=='external')&
 (decisions.capacity==2)&(decisions.reserve==2)&(decisions.reserve_unit_cost==.5)&
 (decisions['mode']=='online_budget_20pct')].sort_values(['region','mean_cost']))


## 12. Figures
Visualize data coverage, held-out count-score comparisons, region-specific forecast timelines, probability calibration and the reserve workload/cost trade-off. All data underlying these figures are saved as CSV tables.

In [ ]:
def finish(fig,name):
    fig.tight_layout();fig.savefig(FIGS/name,dpi=180,bbox_inches='tight');plt.show();plt.close(fig)

qtable=pd.DataFrame(QUALITY)
fig,ax=plt.subplots(figsize=(12,6))
sns.barplot(data=qtable,x='region',y='coverage',ax=ax,errorbar=None)
ax.axhline(CFG['min_coverage'],ls='--',color='red');ax.set_ylim(0,1.05)
ax.set_title('Mean valid weekly coverage; inspect city-level and zero-run audit separately')
finish(fig,'01_coverage.png')

ss=summary[(summary.fold=='final')&(summary.scenario==MAIN)&(summary.horizon==2)]
for evaluation in ['development_temporal','external']:
    shown=ss[ss.evaluation.eq(evaluation)].dropna(subset=['rps'])
    if len(shown):
        fig,ax=plt.subplots(figsize=(11,5))
        sns.barplot(data=shown,x='method',y='rps',ax=ax,errorbar=None)
        ax.tick_params(axis='x',rotation=65);ax.set_title(f'2-week count forecasts: {evaluation}; lower RPS is better')
        finish(fig,f'02_rps_{evaluation}.png')

primary=final[final.horizon.eq(2)]
for region,g in primary.groupby('region'):
    selected=['seasonal_distribution','lr_cal_independent','lr_cal_shared_factor','direct_ordinal_lr']
    template=g[g.method.eq('seasonal_distribution')].sort_values('target_date')
    fig,axes=plt.subplots(2,1,figsize=(13,7),sharex=True)
    axes[0].fill_between(template.target_date,0,template.y,alpha=.15,color='red',label='Observed joint event')
    axes[1].plot(template.target_date,template.n,color='black',lw=1,label='Observed city count')
    for method in selected:
        a=g[g.method.eq(method)].sort_values('target_date')
        if a.empty:continue
        axes[0].plot(a.target_date,a.prob_joint,lw=1,label=method)
        axes[1].plot(a.target_date,a.expected_n,lw=1,label=method)
    axes[0].set(ylabel='P(at least 3 cities)',ylim=(0,1),title=region+' — retrospective revised-data forecasts')
    axes[1].set(ylabel='Cities surging',ylim=(0,10))
    axes[0].legend(fontsize=7,ncol=2);axes[1].legend(fontsize=7,ncol=2)
    finish(fig,f'03_forecast_{region}.png')

bins=pd.DataFrame(bin_records)
for evaluation in ['development_temporal','external']:
    fig,ax=plt.subplots(figsize=(6,5));ax.plot([0,1],[0,1],'k--')
    for method in ['lr_cal_independent','lr_cal_shared_factor','gb_cal_independent','direct_ordinal_lr']:
        g=primary[(primary.evaluation==evaluation)&(primary.method==method)]
        if g.empty:continue
        bb=pd.cut(g.prob_joint,np.linspace(0,1,6),include_lowest=True)
        points=g.groupby(bb,observed=True).agg(predicted=('prob_joint','mean'),observed=('y','mean'))
        ax.plot(points.predicted,points.observed,'o-',label=method)
    ax.set(xlabel='Mean forecast probability',ylabel='Observed event frequency',title=evaluation+' calibration',xlim=(0,1),ylim=(0,1))
    if ax.get_legend_handles_labels()[0]:ax.legend(fontsize=7)
    finish(fig,f'04_calibration_{evaluation}.png')

show=decisions[(decisions.horizon==2)&(decisions.capacity==2)&(decisions.reserve==2)&
  (decisions.reserve_unit_cost==.5)&(decisions.evaluation=='external')]
if len(show):
    fig,ax=plt.subplots(figsize=(10,5))
    sns.scatterplot(data=show,x='activation_rate',y='mean_cost',hue='policy',style='mode',ax=ax)
    ax.set_title('Hypothetical reserve decisions: workload versus cost')
    ax.legend(fontsize=6,bbox_to_anchor=(1.02,1),loc='upper left')
    finish(fig,'05_reserve_cost_workload.png')

rho=status[(status.component=='lr_dependence')&(status.fold=='final')]
if len(rho) and 'rho' in rho:
    rho[['scenario','horizon','available','rho','reason']].to_csv(TABLES/'primary_dependence_estimates.csv',index=False)
print('Figures saved:',len(list(FIGS.glob('*.png'))))


## 13. Assessment report and runtime integrity checks
Review event episodes, calibration availability and uncertainty—not just row counts, AUROC or a visually impressive outbreak curve. This report does not automatically declare success, novelty or publication readiness. Sparse support flags are heuristic planning checks rather than universal sample-size criteria.

In [ ]:
warnings_list=[]
if not CFG['run_external']:warnings_list.append('External geographic evaluation was disabled.')
unavailable=status[status.available.eq(False)]
if len(unavailable):warnings_list.append(f'{len(unavailable)} model/calibration components were unavailable; see model_status.csv.')
for _,r in qtable[qtable.longest_zero_run>=12].iterrows():
    warnings_list.append(f"{r.region}/{r.municipality}: {int(r.longest_zero_run)} consecutive reported-zero weeks; surveillance interpretation unresolved.")
support_main=event_support[(event_support.fold=='final')&(event_support.scenario==MAIN)&event_support.horizon.eq(2)]
for _,r in support_main.iterrows():
    if r.events<20 or r.episodes<5:
        warnings_list.append(f"Support flag {r.region}/{r['split']}: {int(r.events)} event weeks and {int(r.episodes)} episodes (heuristic, not a formal sample-size rule).")
primary_ci=intervals[(intervals.comparison=='primary_matched_dependence')&
   (intervals.evaluation=='external')&intervals.horizon.eq(2)]
eligible=len(primary_ci)>0 and primary_ci.available.all()
lines=['FULL MULTI-REGION DENGUE EXPERIMENT',f'Protocol SHA256: {PROTOCOL_HASH}',
 'Analysis: retrospective downloaded surveillance snapshot, NOT a historical real-time backtest.',
 'Primary question: shared-factor vs independent aggregation of IDENTICAL calibrated logistic city probabilities.',
 'Primary endpoint: equal-region normalized RPS at 2-week horizon in Fortaleza and Recife.',
 'All methods and folds were fixed before external evaluation; no automatic winner selection or tuning.',
 'Rio pilot outcomes were previously reviewed. External histories define local reference thresholds only;',
 'external outcomes do not fit prediction models, calibrators, or the dependence parameter.',
 '',f'Primary external comparison estimable: {eligible}',
 'If not estimable, uncalibrated forecasts are exploratory and do not replace the primary comparison.',
 '', 'PRIMARY EXTERNAL ESTIMATES:']
if len(primary_ci):lines.append(primary_ci.to_string(index=False))
else:lines.append('Unavailable — inspect support and calibration records.')
lines+=['','FIXED FINAL HORIZON-2 SUMMARY:',ss.to_string(index=False),
 '', 'QUALITY/SUPPORT FLAGS:']+warnings_list
lines+=['','LIMITS THAT REMAIN EVEN IF MODELS PERFORM WELL:',
 'Historical cases and population may have been revised. Lag scenarios do not remove revision bias.',
 'Six panels are geographically chosen study panels, not an exhaustive or population-ranked sample.',
 'Only two external regions: uncertainty is conditional on these panels; broad geographic claims remain limited.',
 'Shared-factor aggregation allows positive, exchangeable latent dependence only; it is not a new algorithm.',
 'Ordinal logistic assumes proportional odds; count-class sparsity and convergence are explicitly reported.',
 'Costs, capacities, and equal service units are hypothetical; no observed hospital overload or prevented illness is inferred.',
 'Temporal block CIs condition on fitted models and do not include all training/selection uncertainty.',
 'Few epidemic episodes can dominate results; inspect year, region, and episode omission sensitivity.',
 'No data-revision archive, prospective deployment, clinical validation, or novelty review is supplied by this notebook.',
 'The sensitivity grid does not select the main definition. Do not cherry-pick a favorable scenario.',
 '', 'FILES TO SEND BACK: dengue_full_results.zip and, ideally, the executed Kaggle notebook.',
 'Assess forecasting evidence, calibration, support, uncertainty, operational workload, and whether gains survive external evaluation.']
report='\n'.join(lines)
(OUT/'EXPERIMENT_ASSESSMENT.txt').write_text(report)
print(report)
(OUT/'README.txt').write_text('Read EXPERIMENT_ASSESSMENT.txt first. Raw source snapshots and JSON download metadata are in raw/. The protocol is hashed before data acquisition. Predictions, support, model availability, paired comparisons, reserve simulation, and sensitivity outputs are in tables/. Downloaded historical counts are revised surveillance data. Resource outputs are hypothetical, not clinical outcomes. Source: https://info.dengue.mat.br/tutorial_api_python/')

# Runtime integrity checks on the ACTUAL evaluated data and actions.
assert predictions[['row_key','method','fold','scenario','horizon']].duplicated().sum()==0
for (_,_,_,_),g in city_predictions.groupby(['fold','scenario','horizon','family']):
    assert g[['region','geocode','origin_date']].duplicated().sum()==0
checks=dict(protocol_hash=PROTOCOL_HASH,unique_forecast_keys=True,
            city_regional_target_alignment=True,chronological_availability_purged=True,
            probability_distributions_normalized=True,online_budget_assertions_passed=True,
            assessed_at_utc=datetime.now(timezone.utc).isoformat())
(OUT/'runtime_checks.json').write_text(json.dumps(checks,indent=2))
print('Runtime integrity checks passed.')


## 14. Export and send back
Run this cell even after an earlier failure, provided Setup ran. Download **dengue_full_results.zip** from Kaggle Outputs, and send it together with the executed notebook if possible. Retain the executed notebook using **Save Version → Save & Run All**.

The ZIP contains the frozen protocol, raw downloaded snapshots, timestamp/hash manifests, quality audits, thresholds, forecasts, calibration/model availability, event support, sensitivity results, paired uncertainty, hypothetical resource decisions, figures and assessment report. Nothing is silently substituted with synthetic research data.

In [ ]:
# Safe to run after an earlier error, once Setup has run. Failed runs still export diagnostics.
try:
    files=[]
    for path in sorted(OUT.rglob('*')):
        if path.is_file() and path.name!='file_hashes.csv':
            files.append(dict(file=str(path.relative_to(OUT)),bytes=path.stat().st_size,
                              sha256=hashlib.sha256(path.read_bytes()).hexdigest()))
    pd.DataFrame(files).to_csv(OUT/'file_hashes.csv',index=False)
    zip_path=shutil.make_archive(str(BASE/'dengue_full_results'),'zip',root_dir=OUT)
    print('RESULTS:',zip_path)
    display(FileLink(zip_path))
except Exception:traceback.print_exc()
